# VAA 3차 실험 : F + 진입각 (원본 / 회귀 보정 / 물리 기준점)

2차 실험의 F(구속·무브먼트·회전·릴리스 중심, 궤적 초기값·가속도·진입각 제외)에 진입각을 다시 넣었을 때를 비교한다.

| 변형 | 피처 |
|---|---|
| F | 2차와 같음 (`outputs/vaa_exp/stage3_F.joblib`) |
| H0 | F + `vaa` + `haa` (원본) |
| H1 | F + `vaa_adj` + `haa_adj` (구종별 회귀로 평균 위치 효과를 뺀 값, 1차의 B·D 와 같은 값) |
| H2 | F + `vaa_nom` + `haa_nom` (물리 기준점 : 같은 공을 존 가운데(2.5 ft, 플레이트 중앙)로 던졌다면의 진입각) |
| H3 | F + `vaa_nom` |
| A | 현재 웹 모델 (참고) |

**물리 기준점 진입각** — 릴리스 위치·구속·가속도(무브먼트)만으로 계산, 조준 방향(vx0·vz0)과 실제 도착 위치를 쓰지 않는다
1. t(y) : 50 + vy0·t + ½·ay·t² = y 의 해. t_r = t(release_pos_y), t_p = t(17/12), T = t_p − t_r
2. vz_r = (2.5 − release_pos_z − ½·az·T²) / T,  vz_f = vz_r + az·T,  vy_f = vy0 + ay·t_p
3. vaa_nom = −atan(vz_f / vy_f).  HAA 도 같은 방식, 목표 x = 0 : vx_r = (0 − release_pos_x − ½·ax·T²) / T, vx_f = vx_r + ax·T, haa_nom = −atan(vx_f / vy_f)
- 좌투는 release_pos_x·ax 를 반전한 값(우투 기준)으로 계산

**시간 단축** : 1·2단계(RF OOF)·시즌 로드·기존 모델 채점 결과를 `outputs/vaa_exp/cache/` 에 한 번 저장하고 재사용한다. 새 변형은 F 의 최적 하이퍼파라미터로 1회 학습(선별용).

In [1]:
import os, time, warnings, joblib
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import stuff_pipeline as sp
import export_web_data as ew

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
SEED = sp.SEED
OUT = os.path.join('outputs', 'vaa_exp')
CACHE = os.path.join(OUT, 'cache')
os.makedirs(CACHE, exist_ok=True)
T0 = time.time()

LOC = {'vaa': ('plate_z', 2.5), 'haa': ('plate_x_r', 0.0)}

def num(s):
    return s.to_numpy('float64', na_value=np.nan)

def add_loc(d):
    d['plate_x_r'] = d['plate_x'].where(d['p_throws'] != 'L', -d['plate_x'])
    return d

class LocationAdjuster:
    # 1차 노트북(stuff_pipeline_261010.ipynb)과 같은 정의 : 저장된 계수(location_adjuster.joblib)를 그대로 불러 쓴다
    def transform(self, d):
        for ang, (loc, ref) in LOC.items():
            x = num(d[loc])
            pt = d['pitch_type'].astype(str).to_numpy()
            corr = np.zeros(len(d))
            for p in np.unique(pt):
                c = self.coef_[ang].get(p, self.coef_[ang]['_all'])
                m = pt == p
                corr[m] = np.polyval(c, x[m]) - np.polyval(c, ref)
            d[f'{ang}_adj'] = num(d[ang]) - np.nan_to_num(corr)
        return d

def nominal_angles(d, z_target=2.5, x_target=0.0):
    # 물리 기준점 진입각 (add_stuff_features 이후 = 좌투 반전된 release_pos_x·ax 사용)
    vy0, ay, az, ax = num(d['vy0']), num(d['ay']), num(d['az']), num(d['ax'])
    t_of = lambda y: (-vy0 - np.sqrt(vy0 ** 2 - 2 * ay * (50.0 - y))) / ay
    t_r, t_p = t_of(num(d['release_pos_y'])), t_of(17 / 12)
    T = t_p - t_r
    vy_f = vy0 + ay * t_p
    vz_f = (z_target - num(d['release_pos_z']) - 0.5 * az * T ** 2) / T + az * T
    vx_f = (x_target - num(d['release_pos_x']) - 0.5 * ax * T ** 2) / T + ax * T
    d['vaa_nom'] = -np.degrees(np.arctan(vz_f / vy_f))
    d['haa_nom'] = -np.degrees(np.arctan(vx_f / vy_f))
    return d

## 1. 캐시 (없을 때만 생성 : 1·2단계, 학습 표본, 2024~2026 채점 프레임, A·F 점수)

In [2]:
adjuster = joblib.load(os.path.join(OUT, 'location_adjuster.joblib'))
model_A = joblib.load(os.path.join('models', 'v2', 'stage3_stuff_lgbm.joblib'))
scaler_A = joblib.load(os.path.join('models', 'v2', 'stage4_stuff_scaler.joblib'))
model_F = joblib.load(os.path.join(OUT, 'stage3_F.joblib'))
scaler_F = joblib.load(os.path.join(OUT, 'stage4_F.joblib'))
FEATS = {'A': sp.STUFF_FEATURES_V2,
         'F': [f for f in sp.STUFF_FEATURES if f not in ('vx0', 'vz0', 'ax', 'ay', 'az', 'vy0')]}
cats = model_A.booster_.pandas_categorical
WEB = ['pa_event', 'is_swing', 'is_whiff', 'result', 'xwoba', 'plate_z_norm']
BASE_COLS = ['pitcher', 'pitch_type', 'p_throws', 'game_date', 'description', 'type', 'events', 'estimated_woba_using_speedangle', 'zone',
             'plate_x', 'plate_z', 'plate_x_r', 'release_pos_y', 'vaa', 'haa', 'vaa_adj', 'haa_adj', 'waste']
NEED = sorted(set(sp.STUFF_FEATURES_V2) | set(BASE_COLS) | {'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az'})

def frame(raw):
    d = adjuster.transform(add_loc(sp.add_stuff_features(raw)))
    for col, c in zip(sp.STUFF_CAT_FEATURES, cats):
        d[col] = pd.Categorical(d[col].astype(str), categories=c)
    d = d[d['pitch_type'].notna()]
    d['waste'] = sp.is_waste_ball(d)
    for k, m, s in [('A', model_A, scaler_A), ('F', model_F, scaler_F)]:
        d[f'raw_{k}'] = s.transform(m.predict(d[FEATS[k]]))
    return d

paths = {y: os.path.join(CACHE, f'frame_{y}.parquet') for y in (2024, 2025, 2026)}
if not all(os.path.exists(p) for p in paths.values()):
    t = time.time()
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    df = sp.load_regular_season('games_25_final.pkl')
    train_df, valid_df, valid_start = sp.split_last_month(df, months=1)
    inner_start = valid_start - pd.Timedelta(days=14)
    hit_model = joblib.load(os.path.join('models', 'stage1_hit_rf.joblib'))
    X1_tr, y1_tr = sp.build_hit_dataset(train_df)
    X1_va, _ = sp.build_hit_dataset(valid_df)
    P1_tr = cross_val_predict(RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, max_depth=hit_model.max_depth,
                              min_samples_leaf=hit_model.min_samples_leaf), X1_tr, y1_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                              method='predict_proba')
    calib = sp.GammaBetaCalibrator().fit(sp.expected_value(P1_tr))
    hit_score = pd.concat([pd.Series(calib.transform(sp.expected_value(P1_tr)), index=X1_tr.index),
                           pd.Series(calib.transform(sp.expected_value(hit_model.predict_proba(X1_va))), index=X1_va.index)])
    d = frame(df)
    _, y3 = sp.build_stuff_dataset(d, hit_score, features=sp.STUFF_FEATURES_V2, include_looking_k=False)
    d['y3'] = y3.reindex(d.index)
    gd = d['game_date']
    d['split'] = np.where(gd >= valid_start, 'va', np.where(gd >= inner_start, 'es', 'tr'))
    d = ew.add_web_columns(df, d.assign(**{c: d[c].astype(str) for c in sp.STUFF_CAT_FEATURES}))
    for col, c in zip(sp.STUFF_CAT_FEATURES, cats):
        d[col] = pd.Categorical(d[col], categories=c)
    d[NEED + WEB + ['raw_A', 'raw_F', 'y3', 'split']].to_parquet(paths[2025])
    pd.Series({'valid_start': str(valid_start.date())}).to_json(os.path.join(CACHE, 'meta.json'))
    for year, path in [(2024, 'games_24.pkl'), (2026, 'games_26.pkl')]:
        raw = sp.load_regular_season(path, year=year)
        d = frame(raw)
        d = ew.add_web_columns(raw, d.assign(**{c: d[c].astype(str) for c in sp.STUFF_CAT_FEATURES}))
        for col, c in zip(sp.STUFF_CAT_FEATURES, cats):
            d[col] = pd.Categorical(d[col], categories=c)
        d[NEED + WEB + ['raw_A', 'raw_F']].to_parquet(paths[year])
    print(f'캐시 생성 {time.time() - t:.0f}초')
else:
    print('캐시 사용')

t = time.time()
fr = {y: pd.read_parquet(p) for y, p in paths.items()}
for y in fr:
    fr[y] = nominal_angles(fr[y])
valid_start = pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start']
print(f'캐시 로드 {time.time() - t:.0f}초 :', {y: len(f) for y, f in fr.items()})

캐시 사용
캐시 로드 1초 : {2024: 708479, 2025: 708593, 2026: 713177}


## 2. 물리 기준점 진입각 확인

In [3]:
d25 = fr[2025]
chk = pd.DataFrame({
    'corr(·, plate_z)': [d25[c].corr(d25['plate_z']) for c in ['vaa', 'vaa_adj', 'vaa_nom']],
    'corr(·, plate_x_r)': [d25[h].corr(d25['plate_x_r']) for h in ['haa', 'haa_adj', 'haa_nom']],
}, index=['원본', '회귀 보정', '물리 기준점'])
display(chk.round(3))
g = d25.groupby('pitch_type', observed=True)
pt_tab = pd.DataFrame({'n': g.size(), 'vaa': g['vaa'].mean(), 'vaa_nom': g['vaa_nom'].mean(), 'vaa_nom_sd': g['vaa_nom'].std(),
                       'haa': g['haa'].mean(), 'haa_nom': g['haa_nom'].mean(),
                       'corr_z 원본': g.apply(lambda x: x['vaa'].corr(x['plate_z'])),
                       'corr_z 기준점': g.apply(lambda x: x['vaa_nom'].corr(x['plate_z']))})
display(pt_tab[pt_tab['n'] >= 1000].sort_values('vaa', ascending=False).round(2))

,"corr(·, plate_z)","corr(·, plate_x_r)"
원본,0.754,0.663
회귀 보정,0.298,0.200
물리 기준점,0.294,0.130


,n,vaa,vaa_nom,vaa_nom_sd,haa,haa_nom,corr_z 원본,corr_z 기준점
pitch_type,,,,,,,,
FF,226287,-4.7,-5.11,0.46,1.27,1.27,0.89,-0.12
SI,110228,-5.86,-5.75,0.53,0.47,0.66,0.84,-0.03
FC,53670,-6.12,-6.13,0.64,2.6,2.25,0.85,0.06
CH,73088,-7.46,-6.70,0.58,0.37,0.79,0.85,-0.10
FS,23580,-7.65,-6.82,0.62,0.64,0.94,0.85,-0.09
SL,104854,-7.66,-7.02,0.78,2.92,2.50,0.79,-0.02
ST,49997,-7.71,-7.15,0.81,4.23,3.75,0.78,0.06
SV,3524,-8.63,-7.93,0.74,3.96,3.60,0.80,-0.07
KC,12016,-9.49,-8.58,0.77,2.87,2.71,0.81,-0.14


## 3. 변형 학습 (F 최적 하이퍼파라미터로 1회)

In [4]:
t = time.time()
FEATS.update({
    'H0': FEATS['F'] + ['vaa', 'haa'],
    'H1': FEATS['F'] + ['vaa_adj', 'haa_adj'],
    'H2': FEATS['F'] + ['vaa_nom', 'haa_nom'],
    'H3': FEATS['F'] + ['vaa_nom'],
})
NAMES = {'A': 'A v2 (웹)', 'F': 'F', 'H0': 'H0 F+원본 VAA·HAA', 'H1': 'H1 F+회귀 보정', 'H2': 'H2 F+물리 기준점', 'H3': 'H3 F+기준점 VAA만'}
samp = d25[d25['y3'].notna()]
fit_rows = samp[samp['split'] != 'va']
va_rows = samp[samp['split'] == 'va']
params = model_F.get_params()
models = {'A': model_A, 'F': model_F}
scalers = {'A': scaler_A, 'F': scaler_F}
is_tr = d25['game_date'] < pd.Timestamp(valid_start)
for k in ['H0', 'H1', 'H2', 'H3']:
    m = lgb.LGBMRegressor(**params).fit(fit_rows[FEATS[k]], fit_rows['y3'])
    models[k] = m
    joblib.dump(m, os.path.join(OUT, f'stage3_{k}.joblib'))
    scalers[k] = sp.StuffScaler().fit(m.predict(d25.loc[is_tr & ~d25['waste'], FEATS[k]]))
for y in fr:
    for k in ['H0', 'H1', 'H2', 'H3']:
        fr[y][f'raw_{k}'] = scalers[k].transform(models[k].predict(fr[y][FEATS[k]]))
for y in fr:
    for k in models:
        fr[y][f'stuff_{k}'] = fr[y][f'raw_{k}'].where(~fr[y]['waste'])
print(f'학습·채점 {time.time() - t:.0f}초')

학습·채점 50초


## 4. 지표 (1·2차와 같은 정의, 계산만 벡터화)

In [5]:
t = time.time()
def reg_metrics(y, p):
    return {'RMSE': np.sqrt(mean_squared_error(y, p)), 'MAE': mean_absolute_error(y, p), 'R2': r2_score(y, p)}

def within_corr(d, col, loc, min_n=100):
    # 투수×구종 안 상관의 표본 수 가중 평균 (합계식으로 한 번에)
    x = pd.DataFrame({'g1': d['pitcher'].to_numpy(), 'g2': d['pitch_type'].astype(str).to_numpy(), 'a': num(d[col]), 'b': num(d[loc])}).dropna()
    x = x.assign(ab=x['a'] * x['b'], aa=x['a'] ** 2, bb=x['b'] ** 2)
    s = x.groupby(['g1', 'g2']).agg(n=('a', 'size'), a=('a', 'sum'), b=('b', 'sum'), ab=('ab', 'sum'), aa=('aa', 'sum'), bb=('bb', 'sum'))
    s = s[s['n'] >= min_n]
    cov = s['ab'] - s['a'] * s['b'] / s['n']
    va, vb = s['aa'] - s['a'] ** 2 / s['n'], s['bb'] - s['b'] ** 2 / s['n']
    r = cov / np.sqrt(va * vb)
    ok = r.notna() & np.isfinite(r)
    return np.average(r[ok], weights=s['n'][ok])

def leak_r2(feats, target, n=200_000):
    tr = d25[is_tr & d25[target].notna()].sample(n, random_state=SEED)
    va = d25[~is_tr & d25[target].notna()]
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.08, num_leaves=63, random_state=SEED, verbose=-1).fit(tr[feats], tr[target])
    return r2_score(va[target], m.predict(va[feats]))

def validity(d, col):
    d = d.assign(swing=d['description'].isin(ew.SWING_DESC), whiff=d['description'].isin(sp.WHIFF_DESC),
                 xc=d['estimated_woba_using_speedangle'].where(d['type'] == 'X'))
    pp = d.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum'), xc=('xc', 'mean')).query('n >= 100')
    pt = d.groupby('pitcher').agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 300')
    return {'투수×구종 ρ(구위, whiff%)': stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic,
            '투수×구종 ρ(구위, xwOBAcon)': stats.spearmanr(pp['s'], pp['xc'], nan_policy='omit').statistic,
            '투수 ρ(구위, whiff%)': stats.spearmanr(pt['s'], pt['wh'] / pt['sw']).statistic}

def pt_means(d, col, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str)).groupby(['pitcher', 'pitch_type'])[col].agg(['mean', 'count'])
    return g[g['count'] >= min_n]['mean']

def pt_whiff(d, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str), sw=d['description'].isin(ew.SWING_DESC), wh=d['description'].isin(sp.WHIFF_DESC)) \
         .groupby(['pitcher', 'pitch_type']).agg(n=('sw', 'size'), sw=('sw', 'sum'), wh=('wh', 'sum'))
    g = g[g['n'] >= min_n]
    return g['wh'] / g['sw']

def buckets(d, col):
    bins = pd.cut(d[col], ew.STUFF_BINS, labels=ew.STUFF_BIN_LABELS, right=False)
    return pd.DataFrame({lab: ew.outcome_rates(d[bins == lab]) for lab in ew.STUFF_BIN_LABELS}).T

d26 = fr[2026].assign(abs_x=fr[2026]['plate_x'].abs(), ball=fr[2026]['description'].isin(sp.BALL_DESC),
                      hbp=fr[2026]['description'] == 'hit_by_pitch', out_zone=fr[2026]['zone'] >= 11)
rows = {}
for k in models:
    r = {}
    p = models[k].predict(va_rows[FEATS[k]])
    for a, v in reg_metrics(va_rows['y3'], p).items():
        r[f'검증 {a} (공통 표본)'] = v
    for a, v in validity(d25[~is_tr], f'stuff_{k}').items():
        r[f'검증 {a}'] = v
    for year in (2024, 2026):
        b = buckets(fr[year], f'stuff_{k}')
        r[f'{year} BA 격차'] = b.loc['<40', 'ba'] - b.loc['60+', 'ba']
        r[f'{year} Whiff 격차'] = b.loc['60+', 'whiff'] - b.loc['<40', 'whiff']
    for a, b in [(2024, 2025), (2025, 2026)]:
        m = pd.concat([pt_means(fr[a], f'stuff_{k}'), pt_means(fr[b], f'stuff_{k}')], axis=1, join='inner')
        r[f'안정성 r(구위 {a}, 구위 {b})'] = m.corr().iloc[0, 1]
        w = pd.concat([pt_means(fr[a], f'stuff_{k}'), pt_whiff(fr[b])], axis=1, join='inner')
        r[f'예측력 ρ(구위 {a}, whiff% {b})'] = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1]).statistic
    r['누수 R²(plate_z)'] = leak_r2(FEATS[k], 'plate_z')
    r['누수 R²(plate_x 우투 기준)'] = leak_r2(FEATS[k], 'plate_x_r')
    r['투수×구종 내 corr(점수, 높이 정규화)'] = within_corr(d26, f'stuff_{k}', 'plate_z_norm')
    r['투수×구종 내 corr(점수, |plate_x|)'] = within_corr(d26, f'stuff_{k}', 'abs_x')
    raw = f'raw_{k}'
    top = d26[raw] >= d26[raw].quantile(0.99)
    r['사구 평균 점수 (raw)'] = d26.loc[d26['hbp'], raw].mean()
    r['상위 1% 중 존 밖 볼 비율 (raw)'] = (d26.loc[top, 'ball'] & d26.loc[top, 'out_zone']).mean()
    rows[NAMES[k]] = r
cmp3 = pd.DataFrame(rows)
display(cmp3.round(4))
cmp3.to_csv(os.path.join(OUT, 'comparison_round3.csv'), encoding='utf-8-sig')
print(f'지표 {time.time() - t:.0f}초')

,A v2 (웹),F,H0 F+원본 VAA·HAA,H1 F+회귀 보정,H2 F+물리 기준점,H3 F+기준점 VAA만
검증 RMSE (공통 표본),0.1622,0.1682,0.1627,0.1682,0.1682,0.1681
검증 MAE (공통 표본),0.1131,0.1223,0.1146,0.1222,0.1222,0.1222
검증 R2 (공통 표본),0.0912,0.0222,0.0849,0.0225,0.0221,0.0228
"검증 투수×구종 ρ(구위, whiff%)",0.7383,0.7240,0.7475,0.7121,0.7225,0.7253
"검증 투수×구종 ρ(구위, xwOBAcon)",-0.2860,-0.2432,-0.2632,-0.2394,-0.2371,-0.2433
"검증 투수 ρ(구위, whiff%)",0.5904,0.5485,0.6025,0.5425,0.5528,0.5523
2024 BA 격차,0.2377,0.1141,0.2211,0.1182,0.1149,0.1163
2024 Whiff 격차,0.5469,0.2434,0.5135,0.2456,0.2466,0.2473
2026 BA 격차,0.2405,0.1196,0.2264,0.1220,0.1183,0.1235
2026 Whiff 격차,0.5505,0.2405,0.5147,0.2416,0.2420,0.2405


지표 50초


### 4-1. 재현 검사 : A·F 지표가 2차(comparison_round2.csv)와 같은가

In [6]:
r2 = pd.read_csv(os.path.join(OUT, 'comparison_round2.csv'), index_col=0, encoding='utf-8-sig')
common = [i for i in cmp3.index if i in r2.index]
rep = pd.DataFrame({'A 3차': cmp3.loc[common, NAMES['A']], 'A 2차': r2.loc[common, 'A v2 (원값)'],
                    'F 3차': cmp3.loc[common, NAMES['F']], 'F 2차': r2.loc[common, 'F E + 가속도·vy0 제거']})
rep['A 차이'] = rep['A 3차'] - rep['A 2차']
rep['F 차이'] = rep['F 3차'] - rep['F 2차']
display(rep.round(4))

,A 3차,A 2차,F 3차,F 2차,A 차이,F 차이
검증 RMSE (공통 표본),0.1622,0.1622,0.1682,0.1682,0.0000,0.0000
검증 MAE (공통 표본),0.1131,0.1131,0.1223,0.1223,0.0000,0.0000
검증 R2 (공통 표본),0.0912,0.0912,0.0222,0.0222,0.0000,0.0000
"검증 투수×구종 ρ(구위, whiff%)",0.7383,0.7383,0.7240,0.7240,0.0000,0.0000
"검증 투수×구종 ρ(구위, xwOBAcon)",-0.2860,-0.2860,-0.2432,-0.2432,-0.0000,-0.0000
"검증 투수 ρ(구위, whiff%)",0.5904,0.5904,0.5485,0.5485,0.0000,0.0000
2024 BA 격차,0.2377,0.2377,0.1141,0.1141,0.0000,0.0000
2024 Whiff 격차,0.5469,0.5469,0.2434,0.2434,0.0000,0.0000
2026 BA 격차,0.2405,0.2405,0.1196,0.1196,0.0000,0.0000
2026 Whiff 격차,0.5505,0.5505,0.2405,0.2405,0.0000,0.0000


### 4-2. 구종별 높이 치우침 (투수×구종 내 corr(점수, 높이), 2026) · 요인 그룹 |SHAP|

In [7]:
pts_main = ['FF', 'SI', 'FC', 'SL', 'ST', 'CH', 'FS', 'CU', 'KC']
bias = pd.DataFrame({NAMES[k]: {pt: within_corr(d26[d26['pitch_type'] == pt], f'stuff_{k}', 'plate_z_norm') for pt in pts_main} for k in models})
display(bias.round(3))
bias.to_csv(os.path.join(OUT, 'height_bias_round3.csv'), encoding='utf-8-sig')

X_shap = va_rows.sample(5000, random_state=SEED)
grp = {}
for k, m in models.items():
    c = m.booster_.predict(X_shap[FEATS[k]], pred_contrib=True)[:, :-1]
    names = [f.replace('_adj', '').replace('_nom', '') for f in FEATS[k]]
    grp[NAMES[k]] = {g: (np.abs(c[:, [i for i, f in enumerate(names) if f in cols]].sum(axis=1)).mean() * 1000 if any(f in cols for f in names) else np.nan)
                     for g, cols in ew.EXPLAIN_GROUPS}
display(pd.DataFrame(grp).round(2).rename_axis('평균 |그룹 SHAP| ×1000'))
print(f'전체 실행 {time.time() - T0:.0f}초')

,A v2 (웹),F,H0 F+원본 VAA·HAA,H1 F+회귀 보정,H2 F+물리 기준점,H3 F+기준점 VAA만
FF,0.656,0.037,0.742,0.027,0.041,0.041
SI,0.149,-0.065,0.181,-0.068,-0.055,-0.056
FC,0.099,-0.013,0.073,-0.071,-0.019,-0.026
SL,-0.558,-0.190,-0.651,-0.203,-0.223,-0.217
ST,-0.494,-0.089,-0.581,-0.097,-0.123,-0.108
CH,-0.592,-0.099,-0.636,-0.088,-0.106,-0.106
FS,-0.717,-0.120,-0.772,-0.126,-0.149,-0.148
CU,-0.733,-0.239,-0.824,-0.244,-0.276,-0.271
KC,-0.800,-0.301,-0.857,-0.291,-0.336,-0.331


,A v2 (웹),F,H0 F+원본 VAA·HAA,H1 F+회귀 보정,H2 F+물리 기준점,H3 F+기준점 VAA만
평균 |그룹 SHAP| ×1000,,,,,,
구속,5.24,7.37,7.71,6.52,6.84,7.07
수직 무브먼트,3.51,6.47,6.43,5.81,7.13,7.27
수평 무브먼트,3.20,5.60,4.75,5.15,5.36,5.43
진입각·궤적,35.32,NaN,32.96,4.55,4.49,3.06
릴리스 위치,9.64,3.89,7.47,3.16,3.22,3.55
익스텐션,0.93,1.70,0.77,1.60,1.53,1.49
회전,2.15,3.37,1.97,3.16,3.32,3.30
구종·투구손,5.07,10.75,5.42,11.22,10.93,10.70


전체 실행 153초


## 5. Chamberlain 방식 (VAA Above Average · HAA Above Average) 비교

원문 기준 (FanGraphs, Alex Chamberlain)
- VAA : "normalizing for pitch height … within a specific pitch type" — [A Visualized Primer on Vertical Approach Angle (2022-02-01)](https://blogs.fangraphs.com/a-visualized-primer-on-vertical-approach-angle-vaa/)
- HAA : "adjusted HAA for location, handedness, and release point within each Statcast pitch type" — [A Visual Primer on Horizontal Approach Angle (2023-11-22)](https://blogs.fangraphs.com/a-visual-primer-on-horizontal-approach-angle-haa/)
- 회귀식의 형태는 공개되지 않음 → 가장 단순한 형태로 구현 : 구종별(HAA 는 구종 × 투구손별) **선형회귀 잔차**, 학습기간(2025 검증기간 이전) 투구로 적합
  - vaa_aa = vaa − (a + b·plate_z)
  - haa_aa = haa_raw − (a + b·plate_x + c·release_pos_x)  (투구손별로 따로 적합하므로 좌우 반전 전 원값 사용)

변형 : **H4** = F + vaa_aa + haa_aa, **H5** = F + vaa_aa. 1회 학습(F 하이퍼파라미터)으로 H1(기존 회귀 보정)·H3(물리 기준점 VAA)와 비교

In [8]:
t = time.time()
tr25 = fr[2025][is_tr]

def fit_resid(df_fit, y, xs, keys):
    # 그룹별 선형회귀 계수 {key: beta}
    coef = {}
    for k, g in df_fit.groupby(keys, observed=True):
        g = g[[y] + xs].astype(float).dropna()
        if len(g) < 500:
            continue
        A = np.column_stack([np.ones(len(g))] + [g[x].to_numpy() for x in xs])
        coef[k if isinstance(k, tuple) else (k,)] = np.linalg.lstsq(A, g[y].to_numpy(), rcond=None)[0]
    return coef

def apply_resid(d, y, xs, keys, coef, out):
    res = np.full(len(d), np.nan)
    kv = d[keys].astype(str).apply(tuple, axis=1) if len(keys) > 1 else d[keys[0]].astype(str).map(lambda v: (v,))
    X = np.column_stack([np.ones(len(d))] + [num(d[x]) for x in xs])
    yv = num(d[y])
    for k, b in coef.items():
        m = (kv == tuple(str(v) for v in k)).to_numpy()
        res[m] = yv[m] - X[m] @ b
    d[out] = res
    return d

for y in fr:   # HAA 원값 (좌투 반전 전) : haa 는 우투 기준으로 반전돼 있으므로 좌투는 부호를 되돌린다
    lefty = (fr[y]['p_throws'] == 'L').to_numpy()
    fr[y]['haa_raw'] = np.where(lefty, -num(fr[y]['haa']), num(fr[y]['haa']))
    fr[y]['rel_x_raw'] = np.where(lefty, -num(fr[y]['release_pos_x']), num(fr[y]['release_pos_x']))
tr25 = fr[2025][is_tr]
cv = fit_resid(tr25, 'vaa', ['plate_z'], ['pitch_type'])
ch = fit_resid(tr25, 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'])
for y in fr:
    fr[y] = apply_resid(fr[y], 'vaa', ['plate_z'], ['pitch_type'], cv, 'vaa_aa')
    fr[y] = apply_resid(fr[y], 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'], ch, 'haa_aa')
d25 = fr[2025]
print('구종 안 상관 (2025, 구종별 평균) :')
g = d25.groupby('pitch_type', observed=True)
display(pd.DataFrame({'corr(vaa_aa, plate_z)': g.apply(lambda x: x['vaa_aa'].corr(x['plate_z'])),
                      'corr(vaa_adj, plate_z)': g.apply(lambda x: x['vaa_adj'].corr(x['plate_z'])),
                      'corr(haa_aa, plate_x)': g.apply(lambda x: x['haa_aa'].corr(x['plate_x'])),
                      'n': g.size()}).query('n >= 1000').round(3))

FEATS.update({'H4': FEATS['F'] + ['vaa_aa', 'haa_aa'], 'H5': FEATS['F'] + ['vaa_aa']})
NAMES.update({'H4': 'H4 F+Chamberlain VAA·HAA', 'H5': 'H5 F+Chamberlain VAA만'})
samp = d25[d25['y3'].notna()]
fit_rows, va_rows = samp[samp['split'] != 'va'], samp[samp['split'] == 'va']
for k in ['H4', 'H5']:
    models[k] = lgb.LGBMRegressor(**params).fit(fit_rows[FEATS[k]], fit_rows['y3'])
    joblib.dump(models[k], os.path.join(OUT, f'stage3_{k}.joblib'))
    scalers[k] = sp.StuffScaler().fit(models[k].predict(d25.loc[is_tr & ~d25['waste'], FEATS[k]]))
    for y in fr:
        fr[y][f'raw_{k}'] = scalers[k].transform(models[k].predict(fr[y][FEATS[k]]))
        fr[y][f'stuff_{k}'] = fr[y][f'raw_{k}'].where(~fr[y]['waste'])
d25 = fr[2025]
print(f'학습·채점 {time.time() - t:.0f}초')

구종 안 상관 (2025, 구종별 평균) :


,"corr(vaa_aa, plate_z)","corr(vaa_adj, plate_z)","corr(haa_aa, plate_x)",n
pitch_type,,,,
CH,0.004,0.004,-0.005,73088
CU,0.008,0.007,0.001,49675
FC,0.003,0.002,-0.000,53670
FF,-0.003,-0.003,0.001,226287
FS,0.009,0.009,0.009,23580
KC,-0.007,-0.008,-0.002,12016
SI,-0.001,-0.001,-0.002,110228
SL,-0.001,-0.001,0.002,104854
ST,-0.008,-0.257,-0.005,49997


학습·채점 68초


In [9]:
t = time.time()
d26 = fr[2026].assign(abs_x=fr[2026]['plate_x'].abs(), ball=fr[2026]['description'].isin(sp.BALL_DESC),
                      hbp=fr[2026]['description'] == 'hit_by_pitch', out_zone=fr[2026]['zone'] >= 11)
for k in ['H4', 'H5']:
    r = {}
    p = models[k].predict(va_rows[FEATS[k]])
    for a, v in reg_metrics(va_rows['y3'], p).items():
        r[f'검증 {a} (공통 표본)'] = v
    for a, v in validity(d25[~is_tr], f'stuff_{k}').items():
        r[f'검증 {a}'] = v
    for year in (2024, 2026):
        b = buckets(fr[year], f'stuff_{k}')
        r[f'{year} BA 격차'] = b.loc['<40', 'ba'] - b.loc['60+', 'ba']
        r[f'{year} Whiff 격차'] = b.loc['60+', 'whiff'] - b.loc['<40', 'whiff']
    for a, b in [(2024, 2025), (2025, 2026)]:
        m = pd.concat([pt_means(fr[a], f'stuff_{k}'), pt_means(fr[b], f'stuff_{k}')], axis=1, join='inner')
        r[f'안정성 r(구위 {a}, 구위 {b})'] = m.corr().iloc[0, 1]
        w = pd.concat([pt_means(fr[a], f'stuff_{k}'), pt_whiff(fr[b])], axis=1, join='inner')
        r[f'예측력 ρ(구위 {a}, whiff% {b})'] = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1]).statistic
    r['누수 R²(plate_z)'] = leak_r2(FEATS[k], 'plate_z')
    r['누수 R²(plate_x 우투 기준)'] = leak_r2(FEATS[k], 'plate_x_r')
    r['투수×구종 내 corr(점수, 높이 정규화)'] = within_corr(d26, f'stuff_{k}', 'plate_z_norm')
    r['투수×구종 내 corr(점수, |plate_x|)'] = within_corr(d26, f'stuff_{k}', 'abs_x')
    raw = f'raw_{k}'
    top = d26[raw] >= d26[raw].quantile(0.99)
    r['사구 평균 점수 (raw)'] = d26.loc[d26['hbp'], raw].mean()
    r['상위 1% 중 존 밖 볼 비율 (raw)'] = (d26.loc[top, 'ball'] & d26.loc[top, 'out_zone']).mean()
    cmp3[NAMES[k]] = pd.Series(r)
show = [NAMES[k] for k in ['A', 'F', 'H1', 'H3', 'H4', 'H5']]
display(cmp3[show].round(4))
cmp3.to_csv(os.path.join(OUT, 'comparison_round3.csv'), encoding='utf-8-sig')
bias = pd.DataFrame({NAMES[k]: {pt: within_corr(d26[d26['pitch_type'] == pt], f'stuff_{k}', 'plate_z_norm') for pt in pts_main} for k in ['F', 'H1', 'H3', 'H4', 'H5']})
display(bias.round(3))
print(f'지표 {time.time() - t:.0f}초 / 전체 {time.time() - T0:.0f}초')

,A v2 (웹),F,H1 F+회귀 보정,H3 F+기준점 VAA만,H4 F+Chamberlain VAA·HAA,H5 F+Chamberlain VAA만
검증 RMSE (공통 표본),0.1622,0.1682,0.1682,0.1681,0.1682,0.1681
검증 MAE (공통 표본),0.1131,0.1223,0.1222,0.1222,0.1223,0.1222
검증 R2 (공통 표본),0.0912,0.0222,0.0225,0.0228,0.0218,0.0228
"검증 투수×구종 ρ(구위, whiff%)",0.7383,0.7240,0.7121,0.7253,0.7194,0.7173
"검증 투수×구종 ρ(구위, xwOBAcon)",-0.2860,-0.2432,-0.2394,-0.2433,-0.2468,-0.2461
"검증 투수 ρ(구위, whiff%)",0.5904,0.5485,0.5425,0.5523,0.5385,0.5505
2024 BA 격차,0.2377,0.1141,0.1182,0.1163,0.1137,0.1153
2024 Whiff 격차,0.5469,0.2434,0.2456,0.2473,0.2465,0.2446
2026 BA 격차,0.2405,0.1196,0.1220,0.1235,0.1230,0.1238
2026 Whiff 격차,0.5505,0.2405,0.2416,0.2405,0.2421,0.2402


,F,H1 F+회귀 보정,H3 F+기준점 VAA만,H4 F+Chamberlain VAA·HAA,H5 F+Chamberlain VAA만
FF,0.037,0.027,0.041,0.037,0.036
SI,-0.065,-0.068,-0.056,-0.063,-0.074
FC,-0.013,-0.071,-0.026,-0.058,-0.054
SL,-0.190,-0.203,-0.217,-0.204,-0.201
ST,-0.089,-0.097,-0.108,-0.112,-0.097
CH,-0.099,-0.088,-0.106,-0.096,-0.090
FS,-0.120,-0.126,-0.148,-0.118,-0.119
CU,-0.239,-0.244,-0.271,-0.268,-0.273
KC,-0.301,-0.291,-0.331,-0.322,-0.326


지표 38초 / 전체 259초


### 판정
- 재현 : A·F 지표가 2차(comparison_round2.csv)와 같음 (누수 R² 만 트리 수 축소로 ±0.002). 전체 실행 248초 (캐시 생성 85초 포함, 이전 약 30분)
- **H0 (F + 원본 VAA·HAA)** : 위치 누수가 되살아남. plate_z R² 0.28 → 0.96, 상위 1% 중 존 밖 볼 41% → 89%, 사구 평균 점수 50.7 → 63.7, 구종별 높이 치우침 A 수준(FF +0.74, KC −0.86). 타당성 ρ(whiff%) 0.748 로 가장 높지만 위치 덕분
- **H1 (회귀 보정)·H2 (물리 기준점)·H3 (기준점 VAA 만)** : F 와 거의 같음. ρ(whiff%) 0.712 / 0.723 / 0.725 (F 0.724), 안정성 0.93~0.94, 다음 시즌 whiff% ρ 0.77~0.78, 상위 1% 존 밖 볼 42~43%
  - 위치 중립 진입각에 담긴 정보(릴리스 높이·익스텐션·구속·IVB)는 F 피처에 이미 있어 성능을 더하지 않는다
  - 대신 구위 산출 근거에서 '진입각' 요인으로 설명할 수 있다 (H2·H3 의 진입각 그룹 |SHAP| 3~4.5)
  - H2 의 haa_nom 은 plate_x 누수를 0.18 → 0.43 으로 올림 → H3 (VAA 만)이 더 깔끔
- 결론 : 진입각을 넣는다면 **H3 (F + 위치 중립 VAA)**. 성능은 F 와 같고, 위치 누수·고득점 볼 문제 없이 VAA 를 근거로 보여 줄 수 있다. 채택 전 H3 만 n_iter=12 전체 튜닝으로 확인 필요. 웹 반영은 사용자 결정

**Chamberlain 방식 비교 (5절)**
- 원문 : VAA 는 구종 안에서 투구 높이로, HAA 는 구종 안에서 좌우 위치·투구손·좌우 릴리스 위치로 보정 (회귀식 형태는 비공개 → 구종별 선형 잔차로 구현)
- vaa_aa 와 H1 의 vaa_adj 는 구종 안 높이 상관이 모두 0 으로 사실상 같은 값 (ST 만 차이 : vaa_adj −0.26 = 2차식 적합 문제, vaa_aa −0.01)
- H4 (F + Chamberlain VAA·HAA) · H5 (F + Chamberlain VAA) 성능은 F·H1·H3 와 거의 같음. HAA 를 넣으면 어느 방식이든 plate_x 누수가 0.18 → 0.36~0.43 으로 올라감
- 높이 누수는 H5 0.30 (F 0.28) < H3 0.42 → **VAA 만 넣는다면 Chamberlain 방식(H5)이 조금 더 깔끔**. 타당성 ρ(whiff%) H5 0.717 / H3 0.725, 안정성·예측력은 같은 수준
- 결론 : 진입각 후보는 H5 (F + VAA Above Average, Chamberlain 방식) 또는 H3 (물리 기준점). 둘 다 채택 전 전체 튜닝 확인 필요